# Geneformer perturbation v2: ciliated cell types

**What's different from notebook 01.**
- Inputs are **multi-ciliated and ependymal cells** instead of hepatocytes. TXNDC15 is a ciliogenesis gene (Meckel syndrome); we want to test it in cells where the cilium machinery is actually expressed.
- Gene IDs are **verified by `mygene` lookup** before perturbation, not hand-copied.
- Enrichment includes **explicit overlap with a cilium gene set** (CiliaCarta Gold Standard), not just generic GO.

**Hypothesis.** If TXNDC15's ciliogenesis biology is real and Geneformer has learned it, the TXNDC15 perturbation in ciliated cells should produce a sharper signal than in hepatocytes, and the top affected genes should enrich for axoneme / intraflagellar transport / basal body terms — not lens or pituitary.

**Caveat.** SYVN1 and MARCHF6 are ERAD-only; they may produce weaker signals in ciliated cells than they did in hepatocytes. That's part of the answer.

## 0. Environment setup
If you ran notebook 01 in this session already, you can skip the install cell. If this is a fresh Colab, run it.

In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os
OUT = "/content/drive/MyDrive/benchmate_geneformer_cilia"
os.makedirs(OUT, exist_ok=True)
print("Outputs ->", OUT)

Mounted at /content/drive
Outputs -> /content/drive/MyDrive/benchmate_geneformer_cilia


In [2]:
# Same pinned install as notebook 01. Skip if already done in this session.
!python3 -m pip uninstall -y transformers tokenizers
!python3 -m pip install --no-cache-dir --force-reinstall "transformers==4.40.0" "tokenizers==0.19.1"
!python3 -m pip install --no-cache-dir "peft==0.10.0" "accelerate<1.0" "datasets<3.0"
!python3 -m pip install --no-cache-dir --no-deps git+https://huggingface.co/ctheodoris/Geneformer
!python3 -m pip install --no-cache-dir cellxgene-census gseapy loompy mygene
print("deps installed — restart runtime if you just ran this")

Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
Found existing installation: tokenizers 0.22.2
Uninstalling tokenizers-0.22.2:
  Successfully uninstalled tokenizers-0.22.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.6/137.6 kB 114.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 192.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 314.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 162.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 201.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 236.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 299.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 188.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.2/100.2 kB 264.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━

## 1. Verify gene IDs (no more hand-copy mistakes)

`mygene` queries the canonical NCBI/Ensembl mapping. Whatever it returns here is what we use downstream.

In [3]:
import mygene
mg = mygene.MyGeneInfo()

TARGET_SYMBOLS = ["TXNDC15", "SYVN1", "MARCHF6"]
TARGETS = {}
for sym in TARGET_SYMBOLS:
    hits = mg.query(sym, fields="symbol,ensembl.gene", species="human").get("hits", [])
    for h in hits:
        if h.get("symbol") == sym:
            ens = h.get("ensembl", {})
            ens_id = ens.get("gene") if isinstance(ens, dict) else ens[0]["gene"] if ens else None
            if ens_id:
                TARGETS[sym] = ens_id
                print(f"  {sym} -> {ens_id}")
                break
assert len(TARGETS) == 3, f"Did not resolve all three: {TARGETS}"
print("\nResolved TARGETS:", TARGETS)

  TXNDC15 -> ENSG00000113621
  SYVN1 -> ENSG00000162298
  MARCHF6 -> ENSG00000145495

Resolved TARGETS: {'TXNDC15': 'ENSG00000113621', 'SYVN1': 'ENSG00000162298', 'MARCHF6': 'ENSG00000145495'}


## 2. Pull ciliated cell types from CELLxGENE Census

Filter set:
- `multi-ciliated epithelial cell` — airway epithelium
- `ciliated cell` — generic catch-all
- `ependymal cell` — brain ventricles, motile cilia
- `choroid plexus epithelial cell` — also motile cilia
- `kidney epithelial cell` — primary cilia in tubules

If a label isn't present in the current Census, the filter just returns fewer cells; it won't error.

In [4]:
import cellxgene_census
import numpy as np

CENSUS_VERSION = "2024-07-01"
N_CELLS = 8000

CILIATED_CELL_TYPES = [
    "multi-ciliated epithelial cell",
    "ciliated cell",
    "ependymal cell",
    "choroid plexus epithelial cell",
    "kidney epithelial cell",
    "kidney proximal tubule epithelial cell",
    "kidney distal tubule epithelial cell",
]
cell_type_filter = "cell_type in " + repr(CILIATED_CELL_TYPES)

with cellxgene_census.open_soma(census_version=CENSUS_VERSION) as census:
    adata = cellxgene_census.get_anndata(
        census=census,
        organism="Homo sapiens",
        obs_value_filter=(
            f"{cell_type_filter} "
            "and disease == 'normal' "
            "and is_primary_data == True"
        ),
        column_names={
            "obs": ["cell_type", "tissue", "disease", "assay", "donor_id"],
            "var": ["feature_id", "feature_name"],
        },
    )

print(f"Pulled {adata.n_obs} cells across {adata.obs['cell_type'].nunique()} cell types")
print("\nCell type distribution:")
print(adata.obs["cell_type"].value_counts())
print("\nTissue distribution:")
print(adata.obs["tissue"].value_counts().head(10))

if adata.n_obs > N_CELLS:
    idx = np.random.default_rng(0).choice(adata.n_obs, N_CELLS, replace=False)
    adata = adata[idx].copy()
    print(f"\nDown-sampled to {adata.n_obs} cells")

adata.var["ensembl_id"] = adata.var["feature_id"]
adata.obs["n_counts"] = adata.X.sum(axis=1).A1 if hasattr(adata.X, 'A1') else adata.X.sum(axis=1)

RAW_PATH = f"{OUT}/ciliated_raw.h5ad"
adata.write_h5ad(RAW_PATH)
print("\nsaved:", RAW_PATH)

/tmp/ipykernel_3031/331219770.py:19: FutureWarning: The argument `column_names` is deprecated and will be removed in a future release. Please use `obs_column_names` and `var_column_names` instead.
  adata = cellxgene_census.get_anndata(
/usr/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)
/usr/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


Pulled 74718 cells across 5 cell types

Cell type distribution:
cell_type
ciliated cell                        34505
ependymal cell                       11825
kidney epithelial cell               11116
choroid plexus epithelial cell        9233
multi-ciliated epithelial cell        8039
                                     ...  
endothelial cell of placenta             0
endothelial cell of sinusoid             0
endothelial cell of uterus               0
endothelial cell of vascular tree        0
endosteal cell                           0
Name: count, Length: 698, dtype: int64

Tissue distribution:
tissue
nasal cavity       13812
kidney             10986
cerebral nuclei     7588
trachea             6218
cerebellum          5972
nasopharynx         5381
nose                5171
bronchus            4301
endometrium         3189
myelencephalon      2366
Name: count, dtype: int64

Down-sampled to 8000 cells

saved: /content/drive/MyDrive/benchmate_geneformer_cilia/ciliated_raw.h5ad


## 3. Tokenise + locate the model
Model checkpoint is cached from notebook 01; this is instant on a re-run.

In [8]:
  !pip install -q tdigest
  !pip install -q tdigest anndata loompy pyarrow seaborn statsmodels


In [10]:
  !pip show tdigest 2>&1 | head -5


Name: tdigest
Version: 0.5.2.2
Summary: T-Digest data structure
Home-page: https://github.com/CamDavidsonPilon/tdigest
Author: Cam Davidson-pilon


In [12]:
  !pip install -q optuna


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 29.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
geneformer 0.1.0 requires bitsandbytes, which is not installed.
geneformer 0.1.0 requires optuna-integration, which is not installed.
geneformer 0.1.0 requires ray, which is not installed.


In [13]:
  import importlib
  importlib.invalidate_caches()
  import tdigest
  print("tdigest imports OK:", tdigest.__file__)
  from geneformer import TranscriptomeTokenizer
  print("tokenizer imports OK")

tdigest imports OK: /usr/local/lib/python3.12/dist-packages/tdigest/__init__.py
tokenizer imports OK


In [17]:
  import shutil
  from geneformer import TranscriptomeTokenizer

  # Wipe the old tokenized dataset and re-tokenize with V1 settings
  TOK_IN = "/content/cilia_tok_in"
  TOK_OUT = "/content/cilia_tok_out"
  for d in (TOK_IN, TOK_OUT):
      if os.path.exists(d): shutil.rmtree(d)
      os.makedirs(d)
  shutil.copy(RAW_PATH, f"{TOK_IN}/cilia.h5ad")

  tk = TranscriptomeTokenizer(
      custom_attr_name_dict={"cell_type": "cell_type", "donor_id": "donor_id"},
      nproc=2,
      model_input_size=2048,   # V1 context length
      special_token=False,     # V1 has no <cls>
  )
  tk.tokenize_data(TOK_IN, TOK_OUT, "cilia", file_format="h5ad")
  TOK_DATASET = f"{TOK_OUT}/cilia.dataset"
  print("re-tokenized at 2048-length without CLS")

Tokenizing /content/cilia_tok_in/cilia.h5ad


100%|██████████| 16/16 [01:17<00:00,  4.85s/it]
/usr/local/lib/python3.12/dist-packages/geneformer/tokenizer.py:544: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  for i in adata.var["ensembl_id_collapsed"][coding_miRNA_loc]
/usr/local/lib/python3.12/dist-packages/geneformer/tokenizer.py:547: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  coding_miRNA_ids = adata.var["ensembl_id_collapsed"][coding_miRNA_loc]


/content/cilia_tok_in/cilia.h5ad has no column attribute 'filter_pass'; tokenizing all cells.
Creating dataset.
re-tokenized at 2048-length without CLS


In [15]:
from huggingface_hub import snapshot_download
GENEFORMER_REPO = snapshot_download(repo_id="ctheodoris/Geneformer")
MODEL_PATH = f"{GENEFORMER_REPO}/Geneformer-V1-10M"  # change to V2-104M if on a bigger GPU
print("Model:", MODEL_PATH)

Fetching 87 files:   0%|          | 0/87 [00:00<?, ?it/s]

Geneformer-V1-10M/training_args.bin:   0%|          | 0.00/2.61k [00:00<?, ?B/s]

Geneformer-V1-10M/model.safetensors:   0%|          | 0.00/41.2M [00:00<?, ?B/s]

Geneformer-V1-10M/pytorch_model.bin:   0%|          | 0.00/41.2M [00:00<?, ?B/s]

.pre-commit-config.yaml:   0%|          | 0.00/737 [00:00<?, ?B/s]

...

pretrainer.py: 0.00B [00:00, ?B/s]

tokenizer.py: 0.00B [00:00, ?B/s]

setup.py: 0.00B [00:00, ?B/s]

requirements.txt:   0%|          | 0.00/382 [00:00<?, ?B/s]

Model: /root/.cache/huggingface/hub/models--ctheodoris--Geneformer/snapshots/ad8f66dfcda3ebbd148d916c01f31339c5b95a15/Geneformer-V1-10M


## 4. Perturbation
Same logic as notebook 01, run against ciliated cells instead of hepatocytes.

In [18]:
from geneformer import InSilicoPerturber
import torch, gc

PERTURB_OUT = f"{OUT}/perturbations"
os.makedirs(PERTURB_OUT, exist_ok=True)

def run_perturbation(gene_symbol, ensembl_id):
    out_dir = f"{PERTURB_OUT}/{gene_symbol}"
    os.makedirs(out_dir, exist_ok=True)
    isp = InSilicoPerturber(
        perturb_type="delete",
        genes_to_perturb=[ensembl_id],
        model_type="Pretrained",
        num_classes=0,
        emb_mode="cell_and_gene",
        cell_emb_style="mean_pool",
        filter_data=None,
        cell_states_to_model=None,
        state_embs_dict=None,
        max_ncells=2000,
        emb_layer=0,
        forward_batch_size=4,
        nproc=2,
    )
    isp.perturb_data(
        model_directory=MODEL_PATH,
        input_data_file=TOK_DATASET,
        output_directory=out_dir,
        output_prefix=gene_symbol,
    )
    del isp; gc.collect(); torch.cuda.empty_cache()
    print(f"  -> wrote {gene_symbol} perturbation to {out_dir}")

for sym, eid in TARGETS.items():
    print(f"\nPerturbing {sym} ({eid})...")
    run_perturbation(sym, eid)


Perturbing TXNDC15 (ENSG00000113621)...


  0%|          | 0/150 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/geneformer/perturber_utils.py:380: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)
  sliced_emb = emb[emb_slice]


  0%|          | 0/1 [00:00<?, ?it/s]

...

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  -> wrote MARCHF6 perturbation to /content/drive/MyDrive/benchmate_geneformer_cilia/perturbations/MARCHF6


## 5. Stats + intersection

In [19]:
from geneformer import InSilicoPerturberStats

def run_stats(gene_symbol):
    in_dir = f"{PERTURB_OUT}/{gene_symbol}"
    stats = InSilicoPerturberStats(
        mode="aggregate_gene_shifts",
        genes_perturbed="all",
        combos=0,
        anchor_gene=None,
        cell_states_to_model=None,
    )
    stats.get_stats(
        input_data_directory=in_dir,
        null_dist_data_directory=None,
        output_directory=in_dir,
        output_prefix=f"{gene_symbol}_stats",
    )
    return f"{in_dir}/{gene_symbol}_stats.csv"

stats_files = {sym: run_stats(sym) for sym in TARGETS}
print(stats_files)

  0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

{'TXNDC15': '/content/drive/MyDrive/benchmate_geneformer_cilia/perturbations/TXNDC15/TXNDC15_stats.csv', 'SYVN1': '/content/drive/MyDrive/benchmate_geneformer_cilia/perturbations/SYVN1/SYVN1_stats.csv', 'MARCHF6': '/content/drive/MyDrive/benchmate_geneformer_cilia/perturbations/MARCHF6/MARCHF6_stats.csv'}


In [20]:
import pandas as pd

def load_stats(path, top_n=200, min_detections=200):
    df = pd.read_csv(path)
    if "Affected" in df.columns:
        df = df[df["Affected"] != "cell_emb"].copy()
    df = df.dropna(subset=["Affected_Ensembl_ID"])
    # Filter for hits with enough cell-level support
    if "N_Detections" in df.columns:
        df = df[df["N_Detections"] >= min_detections]
    df = df.sort_values("Cosine_sim_mean", ascending=True)
    return df.head(top_n)

tops = {}
for sym, path in stats_files.items():
    df = load_stats(path)
    tops[sym] = df
    print(f"{sym}: {len(df)} confident gene-level rows, "
          f"top = {df.iloc[0]['Affected_gene_name']} "
          f"(cos_sim={df.iloc[0]['Cosine_sim_mean']:.4f})")

shared3 = set(tops["TXNDC15"]["Affected_Ensembl_ID"])
for sym in ("SYVN1", "MARCHF6"):
    shared3 &= set(tops[sym]["Affected_Ensembl_ID"])
print(f"\n3-way shared (TXNDC15 ∩ SYVN1 ∩ MARCHF6): {len(shared3)}")

# Pairwise to see if TXNDC15 alone drives the cilium signal
pair_txn_syvn = set(tops["TXNDC15"]["Affected_Ensembl_ID"]) & set(tops["SYVN1"]["Affected_Ensembl_ID"])
pair_syvn_mar = set(tops["SYVN1"]["Affected_Ensembl_ID"]) & set(tops["MARCHF6"]["Affected_Ensembl_ID"])
pair_txn_mar  = set(tops["TXNDC15"]["Affected_Ensembl_ID"]) & set(tops["MARCHF6"]["Affected_Ensembl_ID"])
print(f"Pairwise overlaps:")
print(f"  TXNDC15 ∩ SYVN1  : {len(pair_txn_syvn)}")
print(f"  SYVN1   ∩ MARCHF6: {len(pair_syvn_mar)}")
print(f"  TXNDC15 ∩ MARCHF6: {len(pair_txn_mar)}")
print(f"  TXNDC15 only (vs both ERAD genes): {len(set(tops['TXNDC15']['Affected_Ensembl_ID']) - set(tops['SYVN1']['Affected_Ensembl_ID']) - set(tops['MARCHF6']['Affected_Ensembl_ID']))}")

TXNDC15: 200 confident gene-level rows, top = AK9 (cos_sim=0.9640)
SYVN1: 200 confident gene-level rows, top = SOD1 (cos_sim=0.9725)
MARCHF6: 200 confident gene-level rows, top = ZNF814 (cos_sim=0.9455)

3-way shared (TXNDC15 ∩ SYVN1 ∩ MARCHF6): 3
Pairwise overlaps:
  TXNDC15 ∩ SYVN1  : 65
  SYVN1   ∩ MARCHF6: 3
  TXNDC15 ∩ MARCHF6: 15
  TXNDC15 only (vs both ERAD genes): 123


## 6. Enrichment + explicit cilium overlap

Two checks:
1. Standard GO/Reactome/KEGG via Enrichr.
2. Manual overlap with CiliaCarta Gold Standard genes (a curated list of confirmed cilium genes). If TXNDC15's top affected gene set includes >5% of CiliaCarta vs. <1% by chance, the cilium hypothesis has support.

In [21]:
import gseapy as gp

ens_to_sym = dict(zip(adata.var["feature_id"], adata.var["feature_name"]))

def enrich(ens_ids, label):
    syms = [ens_to_sym.get(g, g) for g in ens_ids if g in ens_to_sym]
    if not syms:
        print(f"{label}: no genes mapped")
        return None
    enr = gp.enrichr(
        gene_list=syms,
        gene_sets=["GO_Biological_Process_2023", "Reactome_2022", "KEGG_2021_Human"],
        organism="human",
        outdir=f"{OUT}/enrichment_{label}",
    )
    top = enr.results.sort_values("Adjusted P-value").head(10)[["Gene_set", "Term", "Adjusted P-value", "Genes"]]
    print(f"\n=== Top enrichment for {label} ({len(syms)} genes) ===")
    print(top.to_string(index=False))
    return enr

# Per-perturbation enrichment
for sym in TARGETS:
    enrich(tops[sym]["Affected_Ensembl_ID"].tolist(), sym)

# Intersection enrichment (if non-empty)
if shared3:
    enrich(list(shared3), "shared_3way")
if pair_txn_syvn:
    enrich(list(pair_txn_syvn), "TXNDC15_SYVN1")


=== Top enrichment for TXNDC15 (200 genes) ===
                  Gene_set                                                                                  Term  Adjusted P-value                                                                                                                                                                     Genes
GO_Biological_Process_2023                                                  Cytoplasmic Translation (GO:0002181)      1.021978e-32 RPL3;RPL34;RPLP1;RPLP0;RPL11;RPL8;RPS4X;RPS14;RPS16;RPL13;RPL38;RPL37;RPS11;RPS27A;RPS13;RPL19;RPL41;RPL21;RPS7;RPS8;RPS6;RPL35A;RPS26;RPS28;RPL37A;FAU;RPL29;RPL28;RPS24
             Reactome_2022                                        Eukaryotic Translation Termination R-HSA-72764      1.050459e-30             RPL3;RPL34;RPLP1;RPLP0;RPL11;RPL8;RPS4X;RPS14;RPS16;RPL38;RPL37;RPS11;RPS27A;RPS13;RPL19;RPL41;RPL21;RPS7;RPS8;RPS6;RPL35A;RPS28;RPL37A;FAU;RPL29;RPL28;RPS24
             Reactome_2022                       

In [22]:
# CiliaCarta Gold Standard cilium genes (subset — 50 well-known ciliopathy / axoneme / IFT genes)
# For full list: https://ciliacarta.org/
CILIA_GENES = {
    "IFT88", "IFT80", "IFT172", "IFT122", "IFT43", "IFT57", "IFT81", "IFT74", "IFT20", "IFT52",
    "BBS1", "BBS2", "BBS4", "BBS5", "BBS7", "BBS9", "BBS10", "BBS12",
    "MKS1", "MKS3", "TMEM67", "CC2D2A", "B9D1", "B9D2", "AHI1", "CEP290", "NPHP1", "NPHP3", "NPHP4",
    "ARL13B", "ARL3", "INPP5E", "PKD1", "PKD2", "PKHD1",
    "DNAH5", "DNAH11", "DNAI1", "DNAI2", "DNAAF1", "DNAAF3", "CCDC39", "CCDC40",
    "FOXJ1", "RFX2", "RFX3",
    "TXNDC15", "TBCCD1", "TBCC", "OFD1",
}

def cilium_overlap(ens_ids, label):
    syms = {ens_to_sym.get(g, g) for g in ens_ids}
    overlap = syms & CILIA_GENES
    pct = 100 * len(overlap) / max(len(syms), 1)
    print(f"{label:>20}  : {len(overlap):>3}/{len(syms)} cilium-gene overlap ({pct:.1f}%) -- {sorted(overlap)}")
    return overlap

print("=== Manual CiliaCarta-subset overlap ===")
print("(For random 200-gene lists from human genome, expect ~0.5-1% overlap with this curated set)\n")
for sym in TARGETS:
    cilium_overlap(tops[sym]["Affected_Ensembl_ID"].tolist(), sym)
if shared3:
    cilium_overlap(list(shared3), "shared_3way")

=== Manual CiliaCarta-subset overlap ===
(For random 200-gene lists from human genome, expect ~0.5-1% overlap with this curated set)

             TXNDC15  :   1/200 cilium-gene overlap (0.5%) -- ['IFT88']
               SYVN1  :   2/200 cilium-gene overlap (1.0%) -- ['CC2D2A', 'RFX3']
             MARCHF6  :   0/200 cilium-gene overlap (0.0%) -- []
         shared_3way  :   0/3 cilium-gene overlap (0.0%) -- []


# Section 7: Follow Up

In [23]:

  # 1. Pull the "TXNDC15 specific" gene set — what's TXNDC15 doing
  #    that neither SYVN1 nor MARCHF6 are doing?
  txn_only = set(tops["TXNDC15"]["Affected_Ensembl_ID"]) - \
             set(tops["SYVN1"]["Affected_Ensembl_ID"]) - \
             set(tops["MARCHF6"]["Affected_Ensembl_ID"])
  enrich(list(txn_only), "TXNDC15_only")

  # 2. Pull the "TXNDC15+SYVN1 not MARCHF6" set — the novel-pair axis
  pair_only = (set(tops["TXNDC15"]["Affected_Ensembl_ID"]) &
               set(tops["SYVN1"]["Affected_Ensembl_ID"])) - \
              set(tops["MARCHF6"]["Affected_Ensembl_ID"])
  enrich(list(pair_only), "TXNDC15_SYVN1_only")

  # 3. Print the full 3-way set with names, just to see
  print(tops["TXNDC15"][tops["TXNDC15"]["Affected_Ensembl_ID"].isin(shared3)][
      ["Affected_gene_name", "Affected_Ensembl_ID", "Cosine_sim_mean", "N_Detections"]
  ])


=== Top enrichment for TXNDC15_only (123 genes) ===
                  Gene_set                                                                                  Term  Adjusted P-value                                                                                             Genes
             Reactome_2022                                                Selenocysteine Synthesis R-HSA-2408557      7.063747e-19 RPL3;RPL21;RPS7;RPS8;RPS6;RPL35A;RPL8;RPS14;RPS16;RPL37A;RPL38;RPL37;FAU;RPS11;RPL29;RPS27A;RPL19
             Reactome_2022                                        Eukaryotic Translation Elongation R-HSA-156842      7.063747e-19 RPL3;RPL21;RPS7;RPS8;RPS6;RPL35A;RPL8;RPS14;RPS16;RPL37A;RPL38;RPL37;FAU;RPS11;RPL29;RPS27A;RPL19
             Reactome_2022                                        Eukaryotic Translation Termination R-HSA-72764      7.063747e-19 RPL3;RPL21;RPS7;RPS8;RPS6;RPL35A;RPL8;RPS14;RPS16;RPL37A;RPL38;RPL37;FAU;RPS11;RPL29;RPS27A;RPL19
             Reactome_2022     

## What to look for

- **TXNDC15 cilium overlap >> SYVN1 and MARCHF6 cilium overlap** → TXNDC15-specific ciliogenesis signal is real; the hits in notebook 01 weren't noise.
- **All three show similar cilium overlap (low)** → cilium signal was tissue-mismatch noise in notebook 01.
- **All three show similar cilium overlap (high)** → the model is reading something about cilia in *all* these perturbations, which would be surprising for SYVN1/MARCHF6 and worth investigating.
- **TXNDC15 enrichment includes "intraflagellar transport" / "axoneme" / "ciliary tip" GO terms** → strong support for the ciliogenesis reading.